# 00 · Adquisició de Dades - en procés de revisió i millora
## Immunotherapy Response Prediction — Melanoma (SKCM)

**Objectiu:** Descarregar i integrar dades de tres fonts:
1. **TCGA-SKCM**: RNA-seq (expressió gènica) + clínica + supervivència
2. **cBioPortal**: Mutacions + resposta a immunoteràpia (cohorts anti-PD1)
3. **Kaggle/Literatura**: Datasets suplementaris si escau

---
## 🎯 Justificació de l'elecció: Melanoma + Anti-PD1

| Criteri | Justificació |
|---------|-------------|
| **Dades de resposta real** | cBioPortal té cohorts amb resposta clínica documentada (CR/PR/SD/PD) |
| **Mida cohort TCGA** | TCGA-SKCM: n=472, un dels majors cohorts de melanoma |
| **Biomarcadors coneguts** | TMB, PD-L1, signatura IFN-γ → literature well established |
| **Impacte clínic** | Anti-PD1 és 1a línia estàndard en melanoma avançat |
| **Reproducibilitat** | Dades públiques, sense restriccions d'accés |


In [1]:
import os
import requests
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Directoris
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Entorn configurat correctament')

✅ Entorn configurat correctament


## 1. TCGA-SKCM via GDC API

Usem l'API REST del GDC (Genomic Data Commons) per descarregar:
- **RNA-seq**: HTSeq counts / STAR counts (processats a TPM)
- **Dades clíniques**: edat, sexe, estadi, supervivència

> **Alternativa R**: Si preferiu R, podeu usar:
> ```r
> library(TCGAbiolinks)
> query <- GDCquery(project='TCGA-SKCM', data.category='Transcriptome Profiling',
>                   data.type='Gene Expression Quantification', workflow.type='STAR - Counts')
> GDCdownload(query)
> data <- GDCprepare(query)
> ```

In [2]:
# =============================================================
# TCGA-SKCM: Descàrrega de dades clíniques via GDC REST API
# =============================================================

GDC_BASE = 'https://api.gdc.cancer.gov'

def fetch_tcga_clinical(project='TCGA-SKCM', save_path=None):
    """Descarrega dades clíniques de TCGA via GDC API."""
    
    endpoint = f'{GDC_BASE}/cases'
    params = {
        'filters': json.dumps({
            'op': '=',
            'content': {
                'field': 'project.project_id',
                'value': project
            }
        }),
        'fields': ','.join([
            'case_id', 'submitter_id',
            'demographic.age_at_index',
            'demographic.gender',
            'demographic.race',
            'demographic.vital_status',
            'demographic.days_to_death',
            'diagnoses.age_at_diagnosis',
            'diagnoses.ajcc_pathologic_stage',
            'diagnoses.days_to_last_follow_up',
            'diagnoses.primary_diagnosis',
            'diagnoses.tumor_grade',
            'exposures.pack_years_smoked',
        ]),
        'format': 'JSON',
        'size': '1000'
    }
    
    print(f'📥 Descarregant dades clíniques {project}...')
    response = requests.get(endpoint, params=params)
    response.raise_for_status()
    
    data = response.json()['data']['hits']
    print(f'   → {len(data)} casos trobats')
    
    # Aplanar estructura JSON nested
    records = []
    for case in data:
        record = {
            'case_id': case.get('case_id'),
            'submitter_id': case.get('submitter_id'),
        }
        # Demographics
        demo = case.get('demographic', {})
        record.update({
            'age': demo.get('age_at_index'),
            'gender': demo.get('gender'),
            'race': demo.get('race'),
            'vital_status': demo.get('vital_status'),
            'days_to_death': demo.get('days_to_death'),
        })
        # Diagnosis (agafem el primer)
        diag = case.get('diagnoses', [{}])[0]
        record.update({
            'stage': diag.get('ajcc_pathologic_stage'),
            'days_to_follow_up': diag.get('days_to_last_follow_up'),
            'primary_diagnosis': diag.get('primary_diagnosis'),
        })
        records.append(record)
    
    df = pd.DataFrame(records)
    
    if save_path:
        df.to_csv(save_path, index=False)
        print(f'   → Guardat a {save_path}')
    
    return df

# Executar descàrrega
clinical_df = fetch_tcga_clinical(
    project='TCGA-SKCM',
    save_path=RAW_DIR / 'tcga_skcm_clinical.csv'
)

print(f'\n📊 Shape: {clinical_df.shape}')
print(clinical_df.head())

📥 Descarregant dades clíniques TCGA-SKCM...
   → 470 casos trobats
   → Guardat a ..\data\raw\tcga_skcm_clinical.csv

📊 Shape: (470, 10)
                                case_id  submitter_id   age  gender  \
0  2d36cea4-73e9-493b-8433-0c2b911c0521  TCGA-D3-A3CC  69.0  female   
1  b6691e3c-ee3f-49a2-8bd4-2086413e5d8d  TCGA-DA-A3F8  39.0    male   
2  8ab09143-daf6-40a9-85d3-0fe9de7b3e06  TCGA-ER-A193  62.0    male   
3  b6750b08-c59d-49b9-8c08-a19e17cb366f  TCGA-WE-A8ZO  73.0  female   
4  b6b51b28-bdee-4fe6-a3f3-7b128e16721c  TCGA-GN-A26D  72.0  female   

           race vital_status  days_to_death       stage  days_to_follow_up  \
0         white        Alive            NaN        None             2644.0   
1         white        Alive            NaN        None                NaN   
2         white         Dead          955.0        None                NaN   
3  not reported        Alive            NaN  Stage IIIB                NaN   
4         white         Dead         1460.0   

In [3]:
# =============================================================
# TCGA-SKCM: Descàrrega de dades RNA-seq
# NOTA: Els fitxers RNA-seq de TCGA son grans (~GB).
# Aquí descarreguem els manifest i usem gdc-client.
# =============================================================

def fetch_rnaseq_manifest(project='TCGA-SKCM', save_path=None):
    """Obté manifest de fitxers RNA-seq per descarregar amb gdc-client."""
    
    endpoint = f'{GDC_BASE}/files'
    params = {
        'filters': json.dumps({
            'op': 'and',
            'content': [
                {'op': '=', 'content': {'field': 'cases.project.project_id', 'value': project}},
                {'op': '=', 'content': {'field': 'data_category', 'value': 'Transcriptome Profiling'}},
                {'op': '=', 'content': {'field': 'data_type', 'value': 'Gene Expression Quantification'}},
                {'op': '=', 'content': {'field': 'analysis.workflow_type', 'value': 'STAR - Counts'}},
            ]
        }),
        'fields': 'file_id,file_name,cases.submitter_id,data_type',
        'format': 'JSON',
        'size': '600'
    }
    
    print(f'📥 Obtenint manifest RNA-seq {project}...')
    response = requests.get(endpoint, params=params)
    response.raise_for_status()
    
    files = response.json()['data']['hits']
    print(f'   → {len(files)} fitxers RNA-seq trobats')
    
    manifest = []
    for f in files:
        manifest.append({
            'file_id': f['file_id'],
            'file_name': f['file_name'],
            'case_id': f.get('cases', [{}])[0].get('submitter_id', 'NA')
        })
    
    df_manifest = pd.DataFrame(manifest)
    if save_path:
        df_manifest.to_csv(save_path, index=False)
    
    return df_manifest

manifest = fetch_rnaseq_manifest(save_path=RAW_DIR / 'tcga_skcm_rnaseq_manifest.csv')
print(manifest.head())

# Per descarregar: instal·lar gdc-client i executar:
print('''
📌 Per descarregar els fitxers RNA-seq:

  1. Instal·lar gdc-client:
     https://gdc.cancer.gov/access-data/gdc-data-transfer-tool

  2. Generar manifest a: https://portal.gdc.cancer.gov/
     → Cercar TCGA-SKCM → Transcriptome Profiling → STAR Counts
     → Descarregar manifest.txt

  3. Executar:
     gdc-client download -m manifest.txt -d data/raw/rnaseq/

  4. Alternativa lleugera: Usar FireBrowse o UCSC Xena Browser
     https://xenabrowser.net/datapages/?cohort=TCGA%20Skin%20Cutaneous%20Melanoma%20(SKCM)
''')

📥 Obtenint manifest RNA-seq TCGA-SKCM...
   → 473 fitxers RNA-seq trobats
                                file_id  \
0  2f50c3b2-6e31-4fa8-86b8-44e37496d3a4   
1  ca2eed30-6062-4beb-b9a8-f7034812857a   
2  32c1be5e-e833-43f5-a515-75b6e54dbefb   
3  4da1dea6-f059-49b0-b989-4e0ebd7b18fe   
4  98b164c8-e171-4344-9822-04088aed7705   

                                           file_name       case_id  
0  48155bad-7de9-401d-a495-972cbbb8f276.rna_seq.a...  TCGA-ER-A19B  
1  1fd89d3a-5991-4bfb-b27f-66b5f2a289d8.rna_seq.a...  TCGA-ER-A2NC  
2  22e9b9d1-ac71-43f5-8606-f9f53c6ecc08.rna_seq.a...  TCGA-ER-A2NB  
3  c55bb20b-c929-42e2-be7f-8b1da7bee953.rna_seq.a...  TCGA-ER-A2NH  
4  fe4a0710-a6db-4735-9d1a-113df310def6.rna_seq.a...  TCGA-FS-A4F0  

📌 Per descarregar els fitxers RNA-seq:

  1. Instal·lar gdc-client:
     https://gdc.cancer.gov/access-data/gdc-data-transfer-tool

  2. Generar manifest a: https://portal.gdc.cancer.gov/
     → Cercar TCGA-SKCM → Transcriptome Profiling → STAR Counts


In [9]:
# =============================================================
# ALTERNATIVA LLEUGERA: UCSC Xena Browser (recomanada per setup inicial)
# Descarrega matrius pre-processades directament
# =============================================================

# URLs de descàrrega directa des de UCSC Xena
XENA_URLS = {
    'gene_expression': 'https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.SKCM.sampleMap%2FHiSeqV2_PANCAN.gz',
    'clinical': 'https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/survival%2FSKCM_survival.txt.gz',
    'mutations': 'https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/mc3%2FXENA_TCGA_PanCan.gz'
}

print('📌 URLs per descarregar des de UCSC Xena (més accessible):')
for key, url in XENA_URLS.items():
    print(f'   {key}: {url}')

print('''
💡 Instruccions:
  1. Anar a https://xenabrowser.net/datapages/
  2. Cercar "TCGA Skin Cutaneous Melanoma (SKCM)"
  3. Descarregar:
     - Gene Expression RNAseq → IlluminaHiSeq pancan normalized
     - Phenotype → TCGA survival data
     - Simple Nucleotide Variation → PANCAN SNP calls (MC3)
  4. Guardar a data/raw/
''')

📌 URLs per descarregar des de UCSC Xena (més accessible):
   gene_expression: https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.SKCM.sampleMap%2FHiSeqV2_PANCAN.gz
   clinical: https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/survival%2FSKCM_survival.txt.gz
   mutations: https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/mc3%2FXENA_TCGA_PanCan.gz

💡 Instruccions:
  1. Anar a https://xenabrowser.net/datapages/
  2. Cercar "TCGA Skin Cutaneous Melanoma (SKCM)"
  3. Descarregar:
     - Gene Expression RNAseq → IlluminaHiSeq pancan normalized
     - Phenotype → TCGA survival data
     - Simple Nucleotide Variation → PANCAN SNP calls (MC3)
  4. Guardar a data/raw/



## 2. cBioPortal: Cohorts amb Resposta a Anti-PD1

cBioPortal té cohorts reals de pacients tractats amb immunoteràpia, amb resposta clínica documentada. Usarem:

| Estudi | Descripció | Pacients | Variable resposta |
|--------|-----------|---------|------------------|
| `mel_dfci_2019` | Melanoma anti-PD1, Riaz et al. | 68 | RECIST (CR/PR/SD/PD) |
| `skcm_ucla_2016` | Melanoma pembrolizumab, Hugo et al. | 38 | Resposta binària |
| `mel_msk_2021` | Melanoma, MSK cohort | 321 | Supervivència + resposta |


In [5]:
# =============================================================
# cBioPortal API: Descàrrega de cohorts amb resposta a immunoteràpia
# =============================================================

CBIOPORTAL_BASE = 'https://www.cbioportal.org/api'

def get_cbioportal_studies(keyword='melanoma', save_path=None):
    """Llista estudis disponibles a cBioPortal."""
    response = requests.get(f'{CBIOPORTAL_BASE}/studies', params={'keyword': keyword})
    response.raise_for_status()
    studies = response.json()
    df = pd.DataFrame(studies)[['studyId', 'name', 'description', 'allSampleCount']]
    if save_path:
        df.to_csv(save_path, index=False)
    return df

def get_study_clinical_data(study_id, save_path=None):
    """Descarrega dades clíniques d'un estudi cBioPortal."""
    print(f'📥 Descarregant dades clíniques: {study_id}...')
    
    # Samples
    resp = requests.get(f'{CBIOPORTAL_BASE}/studies/{study_id}/clinical-data',
                        params={'clinicalDataType': 'SAMPLE'})
    resp.raise_for_status()
    
    records = resp.json()
    if not records:
        print(f'   ⚠️  No hi ha dades clíniques per a {study_id}')
        return None
    
    # Pivotejar per tenir una fila per sample
    df = pd.DataFrame(records)
    df_pivot = df.pivot_table(
        index='sampleId', columns='clinicalAttributeId',
        values='value', aggfunc='first'
    ).reset_index()
    
    print(f'   → {len(df_pivot)} mostres, {len(df_pivot.columns)} variables clíniques')
    print(f'   → Variables: {list(df_pivot.columns[:10])}...')
    
    if save_path:
        df_pivot.to_csv(save_path, index=False)
    
    return df_pivot

def get_study_mutations(study_id, gene_list=None, save_path=None):
    """Descarrega mutacions d'un estudi cBioPortal."""
    print(f'📥 Descarregant mutacions: {study_id}...')
    
    # Obtenir molecular profiles
    resp = requests.get(f'{CBIOPORTAL_BASE}/molecular-profiles',
                        params={'studyId': study_id})
    profiles = resp.json()
    
    # Trobar perfil de mutacions
    mut_profile = next((p for p in profiles if p['molecularAlterationType'] == 'MUTATION_EXTENDED'), None)
    
    if not mut_profile:
        print(f'   ⚠️  No hi ha perfil de mutacions per a {study_id}')
        return None
    
    profile_id = mut_profile['molecularProfileId']
    
    # Descarregar mutacions
    resp = requests.get(f'{CBIOPORTAL_BASE}/molecular-profiles/{profile_id}/mutations',
                        params={'sampleListId': f'{study_id}_all'})
    resp.raise_for_status()
    
    mutations = pd.DataFrame(resp.json())
    print(f'   → {len(mutations)} mutacions trobades')
    
    if gene_list:
        mutations = mutations[mutations['gene'].apply(
            lambda g: g.get('hugoGeneSymbol', '') in gene_list if isinstance(g, dict) else False
        )]
    
    if save_path and len(mutations) > 0:
        mutations.to_csv(save_path, index=False)
    
    return mutations

# Llistar estudis de melanoma
studies = get_cbioportal_studies('melanoma')
print('\n📊 Estudis de melanoma disponibles a cBioPortal:')
print(studies.to_string(index=False))

KeyError: "None of [Index(['studyId', 'name', 'description', 'allSampleCount'], dtype='object')] are in the [columns]"

In [ ]:
# Descarregar els 3 cohorts principals d'immunoteràpia

TARGET_STUDIES = [
    'mel_dfci_2019',    # Riaz et al. - principal cohort resposta anti-PD1
    'skcm_ucla_2016',   # Hugo et al. - pembrolizumab
    'mel_msk_2021',     # MSK cohort - gran mida
]

# Gens clau per immunoteràpia
IMMUNE_GENES = [
    'BRAF', 'NRAS', 'NF1', 'CDKN2A', 'PTEN',  # Drivers melanoma
    'TP53', 'RB1', 'ARID1A',                    # Supressors tumorals
    'TMB_NONSYNONYMOUS',                         # TMB proxy
    'B2M', 'HLA-A', 'HLA-B',                    # Presentació antigen
    'STK11', 'KEAP1',                            # Resistència immunoteràpia
]

clinical_data = {}
mutation_data = {}

for study in TARGET_STUDIES:
    print(f'\n{'='*50}')
    print(f'Processant: {study}')
    
    # Dades clíniques
    clin = get_study_clinical_data(
        study,
        save_path=RAW_DIR / f'cbioportal_{study}_clinical.csv'
    )
    if clin is not None:
        clinical_data[study] = clin
    
    # Mutacions
    muts = get_study_mutations(
        study,
        save_path=RAW_DIR / f'cbioportal_{study}_mutations.csv'
    )
    if muts is not None:
        mutation_data[study] = muts

print('\n✅ Descàrrega cBioPortal completada')

## 3. Integració i Definició de la Variable Objectiu

### Variable Objectiu: Resposta a Anti-PD1 (Binària)

Usem criteris RECIST adaptats:
- **Resposta (1)**: CR (Complete Response) + PR (Partial Response)
- **No-resposta (0)**: SD (Stable Disease) + PD (Progressive Disease)

**Proxy per TCGA** (si no hi ha resposta directa):
- OS > 24 mesos en estadis avançats (III/IV) → proxy de possible resposta
- Limitació: no tots els pacients TCGA van rebre immunoteràpia

In [ ]:
# =============================================================
# Definir variable objectiu i integrar datasets
# =============================================================

def define_response_variable(df, study_id):
    """
    Defineix la variable objectiu 'response' (0/1) per a cada cohort.
    Adaptat a les variables disponibles per a cada estudi.
    """
    df = df.copy()
    
    # Mapeig de columnes comunes de resposta en cBioPortal
    response_cols = [
        'BEST_RESPONSE', 'OVERALL_RESPONSE', 'RECIST_RESPONSE',
        'RESPONSE', 'TREATMENT_RESPONSE', 'OBJECTIVE_RESPONSE'
    ]
    
    found_col = None
    for col in response_cols:
        if col in df.columns:
            found_col = col
            break
    
    if found_col:
        print(f'   ✓ Columna de resposta trobada: {found_col}')
        print(f'   Valors únics: {df[found_col].unique()}')
        
        # Mapeig RECIST → binari
        response_map = {
            'CR': 1, 'Complete Response': 1, 'COMPLETE_RESPONSE': 1,
            'PR': 1, 'Partial Response': 1, 'PARTIAL_RESPONSE': 1,
            'SD': 0, 'Stable Disease': 0, 'STABLE_DISEASE': 0,
            'PD': 0, 'Progressive Disease': 0, 'PROGRESSIVE_DISEASE': 0,
            'NE': np.nan, 'Not Evaluable': np.nan,
        }
        df['response'] = df[found_col].map(response_map)
        
    else:
        print(f'   ⚠️ Cap columna de resposta directa trobada. Usant proxy de supervivència.')
        print(f'   Variables disponibles: {list(df.columns)}')
        
        # Proxy: OS_STATUS + OS_MONTHS
        if 'OS_STATUS' in df.columns and 'OS_MONTHS' in df.columns:
            df['OS_MONTHS'] = pd.to_numeric(df['OS_MONTHS'], errors='coerce')
            df['response'] = ((df['OS_STATUS'] == '1:DECEASED') & 
                              (df['OS_MONTHS'] > 24)).astype(float)
            print('   → Proxy: OS > 24 mesos entre pacients amb event')
        else:
            df['response'] = np.nan
            print('   ❌ No s\'ha pogut definir la variable objectiu')
    
    n_resp = df['response'].sum()
    n_total = df['response'].notna().sum()
    print(f'   → Respostes: {n_resp}/{n_total} ({100*n_resp/n_total:.1f}%)')
    
    return df


# Aplicar a cada cohort
processed_cohorts = {}
for study, df in clinical_data.items():
    print(f'\nProcessant variable objectiu per {study}:')
    processed_cohorts[study] = define_response_variable(df, study)

# Guardar datasets processats
for study, df in processed_cohorts.items():
    df.to_csv(PROCESSED_DIR / f'{study}_with_response.csv', index=False)

print('\n✅ Datasets amb variable objectiu guardats')

In [ ]:
# Resum final dels datasets disponibles
print('='*60)
print('RESUM FINAL DE DADES ADQUIRIDES')
print('='*60)

datasets_summary = {
    'Font': ['TCGA-SKCM', 'mel_dfci_2019', 'skcm_ucla_2016', 'mel_msk_2021'],
    'Tipus dades': ['RNA-seq + Clínica', 'Clínica + Mutacions', 'Clínica + RNA-seq', 'Clínica + Mutacions'],
    'Resposta directa': ['No (proxy OS)', 'Sí (RECIST)', 'Sí (binària)', 'Sí (parcial)'],
    'Ús en projecte': ['Feature engineering (RNA-seq)', 'Model principal', 'Validació', 'Dataset extens'],
}

summary_df = pd.DataFrame(datasets_summary)
print(summary_df.to_string(index=False))

print('''
📌 ESTRATÈGIA FINAL:

  Dataset PRINCIPAL: mel_dfci_2019 (Riaz et al.)
  - Té resposta clínica directa (RECIST)
  - Dades transcriptòmiques pre/post tractament
  - Ben caracteritzat a la literatura

  Dataset VALIDACIÓ: skcm_ucla_2016 (Hugo et al.)
  - Validació independent del model
  - Resposta a pembrolizumab (anti-PD1)

  TCGA-SKCM:
  - Font d'expressió gènica per calcular signatures
  - Pre-entrenament / transfer learning de signatures immune
''')